In [1]:
import joblib
model = joblib.load("house_price_model.pkl")

In [10]:
import pandas as pd
test_data = pd.read_csv(r"C:\Users\DELL\Desktop\Nest_Internship\house_price_predic\test.csv")
X_test = test_data  

In [12]:
# 1. One-hot encode the test data to turn categorical variables into columns
X_test_encoded = pd.get_dummies(X_test)

# 2. Extract the exact features your model was trained on
model_features = model.feature_names_in_

# 3. Align the columns: adds missing columns as 0, drops columns the model doesn't know
X_test_final = X_test_encoded.reindex(columns=model_features, fill_value=0)

# 4. FIX: Fill any remaining NaN values with 0 so the Gradient Boosting model doesn't crash
X_test_final = X_test_final.fillna(0)

# 5. Generate the predictions
predictions = model.predict(X_test_final)

# 6. Display the first 5 predictions to confirm it works!
print("Predictions successful!")
print(predictions[:5])

Predictions successful!
[310293.28570178 316888.4652747  336984.60665253 341815.00499777
 347021.26225869]


In [14]:
# Create a clean DataFrame to display the predictions with House Id
output_table = pd.DataFrame({
    'Id': X_test['Id'].iloc[:5],
    'Predicted_Price': predictions[:5]
})

# Format the price column as currency for better readability
output_table['Predicted_Price'] = output_table['Predicted_Price'].map('${:,.2f}'.format)

# Display the table
output_table

,Id,Predicted_Price
0,1461,"$310,293.29"
1,1462,"$316,888.47"
2,1463,"$336,984.61"
3,1464,"$341,815.00"
4,1465,"$347,021.26"


In [15]:
# 1. Load the training data (which actually has the true prices)
train_data = pd.read_csv(r"C:\Users\DELL\Desktop\Nest_Internship\house_price_predic\train.csv")

# 2. Get predictions on the training data using the same features
X_train_encoded = pd.get_dummies(train_data.drop(columns=['SalePrice']))
X_train_final = X_train_encoded.reindex(columns=model_features, fill_value=0).fillna(0)
train_predictions = model.predict(X_train_final)

# 3. Create a comparison table for the first 5 houses
comparison_table = pd.DataFrame({
    'Id': train_data['Id'].iloc[:5],
    'Actual_Price': train_data['SalePrice'].iloc[:5],
    'Predicted_Price': train_predictions[:5]
})

# 4. Format both columns as currency
comparison_table['Actual_Price'] = comparison_table['Actual_Price'].map('${:,.2f}'.format)
comparison_table['Predicted_Price'] = comparison_table['Predicted_Price'].map('${:,.2f}'.format)

comparison_table

,Id,Actual_Price,Predicted_Price
0,1,"$208,500.00","$341,815.00"
1,2,"$181,500.00","$336,984.61"
2,3,"$223,500.00","$341,815.00"
3,4,"$140,000.00","$333,133.95"
4,5,"$250,000.00","$341,815.00"


In [19]:
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. RELOAD TRAINING DATA TO DEFINE 'df' AND 'scaler'
# =====================================================================
# FIXED: Updated path to point to your DELL user profile
df = pd.read_csv(r"C:\Users\DELL\Desktop\Nest_Internship\house_price_predic\train.csv")

# The exact columns your model expects
important_num_cols = ['OverallQual', 'YearBuilt', 'YearRemodAdd', 'TotalBsmtSF', '1stFlrSF', 'GrLivArea', 'FullBath', 'TotRmsAbvGrd', 'GarageCars', 'GarageArea']
cat_cols = ["MSZoning", "Utilities", "BldgType", "Heating", "KitchenQual", "SaleCondition", "LandSlope"]

# Re-create X training columns to use as a structural template
X_train_blueprint = df[important_num_cols + cat_cols]
X_train_encoded = pd.get_dummies(X_train_blueprint, columns=cat_cols)

# Fit the scaler on your original training numbers
scaler = StandardScaler()
scaler.fit(X_train_blueprint[important_num_cols])

# =====================================================================
# 2. LOAD SAVED MODEL AND TEST DATA
# =====================================================================
# Make sure this model file is in your current folder, or update its full path if needed
loaded_model = joblib.load('house_price_model.pkl') 

# FIXED: Updated path to point to your DELL user profile
new_data = pd.read_csv(r"C:\Users\DELL\Desktop\Nest_Internship\house_price_predic\test.csv")

# Filter test data to keep only required features
new_data_filtered = new_data[important_num_cols + cat_cols].copy()

# Handle potential missing values (NaNs) in the test set using training averages
for col in important_num_cols:
    new_data_filtered[col] = new_data_filtered[col].fillna(df[col].mean())

# =====================================================================
# 3. PREPROCESS TEST DATA
# =====================================================================
# One-hot encode test data
new_data_encoded = pd.get_dummies(new_data_filtered, columns=cat_cols)

# Align test columns perfectly with training columns blueprint
new_data_encoded = new_data_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Scale numerical features using the fitted scaler instance
new_data_encoded[important_num_cols] = scaler.transform(new_data_filtered[important_num_cols])

# =====================================================================
# 4. PREDICT & DISPLAY AS A TABLE
# =====================================================================
predicted_prices = loaded_model.predict(new_data_encoded)

results = pd.DataFrame({
    'House_Id': new_data['Id'],
    'Predicted_SalePrice': predicted_prices
})

# Format the price column as currency
results['Predicted_SalePrice'] = results['Predicted_SalePrice'].map('${:,.2f}'.format)

# Display the first 5 rows of the table
print(results)

      House_Id Predicted_SalePrice
0         1461         $115,593.91
1         1462         $154,572.32
2         1463         $161,648.09
3         1464         $181,546.06
4         1465         $211,224.22
...        ...                 ...
1454      2915          $79,171.70
1455      2916          $82,615.40
1456      2917         $145,009.00
1457      2918         $119,073.51
1458      2919         $251,071.51

[1459 rows x 2 columns]


In [21]:
# 1. Prepare and scale the TRAINING data so it matches what the model expects
X_train_blueprint = df[important_num_cols + cat_cols]
X_train_encoded = pd.get_dummies(X_train_blueprint, columns=cat_cols)
X_train_encoded = X_train_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

X_train_scaled = X_train_encoded.copy()
X_train_scaled[important_num_cols] = scaler.transform(X_train_blueprint[important_num_cols])

# 2. FIXED: Generate predictions using the training data (1460 rows)
train_predictions = loaded_model.predict(X_train_scaled)

# 3. Create a clean dataframe comparing actual and predicted values (Both now have 1460 rows)
comparison_table = pd.DataFrame({
    'House_Id': df['Id'],
    'Actual_SalePrice': df['SalePrice'],
    'Predicted_SalePrice': train_predictions
})

# 4. Format both columns as currency for easy reading
comparison_table['Actual_SalePrice'] = comparison_table['Actual_SalePrice'].map('${:,.2f}'.format)
comparison_table['Predicted_SalePrice'] = comparison_table['Predicted_SalePrice'].map('${:,.2f}'.format)

# 5. Display the first 10 rows to compare
comparison_table.head(10)

,House_Id,Actual_SalePrice,Predicted_SalePrice
0,1,"$208,500.00","$197,173.20"
1,2,"$181,500.00","$158,168.30"
2,3,"$223,500.00","$209,332.52"
3,4,"$140,000.00","$168,160.21"
4,5,"$250,000.00","$281,870.34"
5,6,"$143,000.00","$146,294.58"
6,7,"$307,000.00","$269,467.44"
7,8,"$200,000.00","$216,296.61"
8,9,"$129,900.00","$148,908.17"
9,10,"$118,000.00","$117,419.94"
